# Distran: Kaggle metadata, scores and content training
Three user-supplied datasets are downloaded through kagglehub with actual filenames. CPU execution. See KAGGLE_TRAINING_RESULTS.md alongside this notebook. This run evaluates aggregate-score regression with title-group holdouts, exports weighted aggregate ranking baselines, and fits a shared content model. None of the files contains individual user preference events.

Reference inspected: local notebook `notebookca2a57168c.ipynb`, "The Age of Recommender Systems". We adapt its TF-IDF and Bayesian weighted-rating ideas. Its TMDB credits join and Surprise collaborative section cannot be applied to these schemas; its old Surprise API is not executed. Reference SHA256: 2948a8c6da02d1fb6f412e7b499aacb646bb1ca3e804e6c680f7fce9fd236f42

In [ ]:
# @title Pin dependencies
import sys,subprocess
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","numpy==2.2.6","pandas==2.2.3","scikit-learn==1.6.1","kagglehub[pandas-datasets]==0.4.3"])

0

In [ ]:
# @title Install the exact experiment and app snapshot
from pathlib import Path
import sys,json
root=Path("/content/distran-metadata")
root.mkdir(parents=True,exist_ok=True)
files={'metadata_experiment.py': '"""Kaggle metadata training: content retrieval, aggregate-score regression, and app vectors."""\nimport hashlib,json,re,sys,time,unicodedata,subprocess\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import mean_absolute_error,mean_squared_error\nfrom sklearn.preprocessing import normalize\n\nHANDLES={\n \'movies\':\'balaka18/tmdb-top-10000-popular-movies-dataset/versions/2\',\n \'sales\':\'rush4ratio/video-game-sales-with-ratings\',\n \'hltb\':\'zaireali/howlongtobeat-games-scraper-2162025/versions/1\'}\nFILES={\'movies\':\'movies_tmdb_popular.csv\',\'sales\':\'Video_Games_Sales_as_at_22_Dec_2016.csv\',\'hltb\':\'hltb_data.csv\'}\n\ndef clean(value):\n    if pd.isna(value):return \'\'\n    return re.sub(r\'\\s+\',\' \',str(value)).strip()\n\ndef title_key(value):\n    value=unicodedata.normalize(\'NFKD\',clean(value)).encode(\'ascii\',\'ignore\').decode().lower()\n    # Apostrophes are omitted; other punctuation becomes spaces. Keep sequel numerals.\n    value=value.replace("\'",\'\')\n    return re.sub(r\'[^a-z0-9]+\',\' \',value).strip()\n\ndef split_label(title):\n    bucket=int(hashlib.sha256(title_key(title).encode()).hexdigest()[:8],16)%10\n    return \'train\' if bucket<6 else \'validation\' if bucket<8 else \'test\'\n\ndef year_value(value):\n    match=re.match(r\'^(19\\d{2}|20\\d{2})\',clean(value))\n    return int(match[0]) if match else None\n\ndef weighted_scores(frame,score,count):\n    ratings=pd.to_numeric(frame[score],errors=\'coerce\');votes=pd.to_numeric(frame[count],errors=\'coerce\').clip(lower=0)\n    valid=ratings.notna() & votes.gt(0)\n    prior=float(ratings[valid].mean()) if valid.any() else 0.\n    m=float(votes[valid].quantile(.9)) if valid.any() else 1.\n    quality=(votes*ratings+m*prior)/(votes+m)\n    return quality.where(valid),dict(prior_mean=prior,prior_count=m,formula=\'v/(v+m)*R + m/(v+m)*C\',eligible=int(valid.sum()))\n\ndef prepare_frames(raw):\n    movies=raw[\'movies\'].copy();movies=movies[movies[\'title\'].notna()].copy()\n    movies[\'key\']=movies[\'title\'].map(title_key);movies[\'year\']=movies[\'rel_date\'].map(year_value)\n    movies=movies.drop_duplicates([\'key\',\'year\']).reset_index(drop=True)\n    movies[\'doc\']=movies[\'overview\'].map(clean)+\' \'+movies[\'original_lang\'].map(lambda x:\'language_\'+clean(x))\n    movies[\'title_doc\']=movies[\'title\'].map(clean)+\' \'+movies[\'doc\']\n    sales=raw[\'sales\'].copy();sales=sales[sales[\'Name\'].notna()].copy()\n    sales[\'key\']=sales[\'Name\'].map(title_key);sales[\'year\']=sales[\'Year_of_Release\'].map(year_value)\n    sales=sales.drop_duplicates([\'key\',\'Platform\',\'year\']).reset_index(drop=True)\n    # Platform, ratings and sales are not included in shared semantic space.\n    sales[\'doc\']=sales[[\'Genre\',\'Publisher\',\'Developer\']].fillna(\'\').astype(str).agg(\' \'.join,axis=1)\n    sales[\'title_doc\']=sales[\'Name\'].map(clean)+\' \'+sales[\'doc\']\n    hltb=raw[\'hltb\'].copy();hltb=hltb[hltb[\'game_game_name\'].notna()].copy()\n    # Do not sum repeated platform rows of game-level aggregates.\n    hltb=hltb.sort_values([\'game_game_id\',\'platform_platform\'],kind=\'stable\').drop_duplicates(\'game_game_id\').copy()\n    hltb[\'key\']=hltb[\'game_game_name\'].map(title_key)\n    hltb[\'year\']=hltb[\'game_release_world\'].map(year_value)\n    hltb[\'doc\']=hltb[[\'game_profile_summary\',\'game_profile_genre\',\'game_profile_dev\',\'game_profile_pub\']].fillna(\'\').astype(str).agg(\' \'.join,axis=1)\n    hltb[\'title_doc\']=hltb[\'game_game_name\'].map(clean)+\' \'+hltb[\'doc\']\n    # One unambiguous key/year on each side; no title-only or fuzzy joins.\n    s_keys=sales[[\'key\',\'year\']].dropna().drop_duplicates()\n    h_keys=hltb[[\'key\',\'year\']].dropna()\n    unique_h=h_keys[~h_keys.duplicated([\'key\',\'year\'],keep=False)]\n    proposals=s_keys.merge(unique_h,on=[\'key\',\'year\'])\n    audits=dict(score_ranges={k:{c:dict(nonmissing=int(pd.to_numeric(v[c],errors=\'coerce\').notna().sum()),minimum=float(pd.to_numeric(v[c],errors=\'coerce\').min()),maximum=float(pd.to_numeric(v[c],errors=\'coerce\').max())) for c in ({\'movies\':[\'vote_average\',\'vote_count\',\'popularity\'],\'sales\':[\'Critic_Score\',\'User_Score\',\'Critic_Count\',\'User_Count\'],\'hltb\':[\'game_review_score\',\'game_count_review\',\'game_comp_main\',\'game_comp_main_count\']}[k])} for k,v in raw.items()},raw_rows={k:len(v) for k,v in raw.items()},processed_rows=dict(movies=len(movies),sales=len(sales),hltb=len(hltb)),missing={k:{c:int(v[c].isna().sum()) for c in v.columns} for k,v in raw.items()},sales_tbd_user_scores=int(raw[\'sales\'][\'User_Score\'].astype(str).str.lower().eq(\'tbd\').sum()),exact_title_year_game_link_proposals=len(proposals),movie_genre_policy=\'genres(1).csv is a standalone dictionary with no per-movie foreign keys; NOT attached\',game_identity_policy=\'Exact normalized title + exact release year only; proposals are not verified canonical mappings. Platform variants retain rows in regression but stay together in title-group splits.\',hltb_policy=\'Drop repeated game_game_id platform rows; use game-level summary/genres, never review histograms or platform completion fields as predictors of aggregate scores.\')\n    return movies,sales,hltb,audits\n\ndef regression(frame,title,target,features,output,name):\n    y=pd.to_numeric(frame[target],errors=\'coerce\')\n    maximum=10 if name==\'movie-votes\' else 100\n    frame=frame[y.between(0,maximum)].copy();frame[\'target\']=y[y.between(0,maximum)]\n    labels=frame[title].map(split_label)\n    masks={x:labels.eq(x).to_numpy() for x in [\'train\',\'validation\',\'test\']}\n    assert not (set(frame.loc[masks[\'train\'],title].map(title_key)) & set(frame.loc[masks[\'test\'],title].map(title_key)))\n    docs=frame[features].fillna(\'\').astype(str).agg(\' \'.join,axis=1).tolist()\n    tf=TfidfVectorizer(stop_words=\'english\',ngram_range=(1,2),max_features=20000,min_df=2,sublinear_tf=True)\n    X=tf.fit_transform(np.array(docs)[masks[\'train\']]);y=frame[\'target\'].to_numpy()\n    val=tf.transform(np.array(docs)[masks[\'validation\']]);test=tf.transform(np.array(docs)[masks[\'test\']])\n    candidates={};trained={}\n    for alpha in [1.,10.,100.]:\n        model=Ridge(alpha=alpha,solver=\'lsqr\');model.fit(X,y[masks[\'train\']]);pred=np.clip(model.predict(val),0,maximum)\n        candidates[str(alpha)]=float(mean_absolute_error(y[masks[\'validation\']],pred));trained[alpha]=model\n    alpha=float(min(candidates,key=candidates.get));model=trained[alpha]\n    pred=np.clip(model.predict(test),0,maximum);truth=y[masks[\'test\']]\n    mean=float(y[masks[\'train\']].mean());baseline=np.full(len(truth),mean)\n    metrics=lambda p:dict(mae=float(mean_absolute_error(truth,p)),rmse=float(np.sqrt(mean_squared_error(truth,p))))\n    data=dict(task=\'aggregate score regression; NOT personal preference or recommendation accuracy\',target=target,range=[0,maximum],features=features,split=\'SHA256(normalized title) modulo 10: 0-5 train, 6-7 validation, 8-9 test; all platform and remake title variants stay together; NOT chronological\',counts={x:int(m.sum()) for x,m in masks.items()},vocabulary=len(tf.vocabulary_),alpha=alpha,validation_mae=candidates,test=metrics(pred),train_mean_baseline=metrics(baseline),train_mean=mean,prediction_clip=[0,maximum])\n    (output/(name+\'-regressor.json\')).write_text(json.dumps(dict(vocabulary=tf.vocabulary_,idf=tf.idf_.tolist(),coefficients=model.coef_.tolist(),intercept=float(model.intercept_),parameters=data),default=int))\n    print(\'REGRESSION\',name,data,flush=True)\n    return data\n\ndef run(root=Path(\'/content/distran-metadata\')):\n    import kagglehub\n    import sklearn\n    start=time.time();root.mkdir(exist_ok=True,parents=True);output=root/\'artifacts\';output.mkdir(exist_ok=True)\n    raw={};sources={}\n    for key,handle in HANDLES.items():\n        directory=Path(kagglehub.dataset_download(handle));path=directory/FILES[key]\n        raw[key]=pd.read_csv(path,low_memory=False)\n        sources[key]=dict(handle=handle,resolved_path=str(directory),file=path.name,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),bytes=path.stat().st_size,rows=len(raw[key]),columns=raw[key].columns.tolist())\n        print(\'LOADED\',key,len(raw[key]),flush=True)\n    movies,sales,hltb,audit=prepare_frames(raw)\n    # Group-safe heldout prediction using metadata only; scores/counts excluded from predictors.\n    regressions={\n      \'movies\':regression(movies[pd.to_numeric(movies[\'vote_count\'],errors=\'coerce\').ge(5)],\'title\',\'vote_average\',[\'overview\',\'original_lang\'],output,\'movie-votes\'),\n      \'sales\':regression(sales,\'Name\',\'Critic_Score\',[\'Genre\',\'Platform\',\'Publisher\',\'Developer\',\'Rating\'],output,\'game-critics\'),\n      \'hltb\':regression(hltb[pd.to_numeric(hltb[\'game_count_review\'],errors=\'coerce\').ge(5)],\'game_game_name\',\'game_review_score\',[\'game_profile_summary\',\'game_profile_genre\',\'game_profile_dev\',\'game_profile_pub\'],output,\'hltb-reviews\')}\n    print(\'Fitting catalog semantic model\',flush=True)\n    # Bound total game dominance and repeated platform copies in the shared content corpus.\n    sales_docs=sales.drop_duplicates([\'key\',\'doc\'])\n    hltb_selected=hltb.assign(sample_hash=hltb[\'key\'].map(lambda k:hashlib.sha256(k.encode()).hexdigest())).sort_values([\'sample_hash\',\'game_game_id\']).head(10000)\n    corpus=list(movies[\'title_doc\'])+list(sales_docs[\'title_doc\'])+list(hltb_selected[\'title_doc\'])\n    tf=TfidfVectorizer(stop_words=\'english\',ngram_range=(1,2),max_features=30000,min_df=2,sublinear_tf=True)\n    X=tf.fit_transform(corpus)\n    svd=TruncatedSVD(n_components=64,n_iter=5,random_state=42);svd.fit(X)\n    catalog=json.loads((root/\'catalog-metadata.json\').read_text())\n    app_docs=[\' \'.join([w[\'title\'],w[\'description\'],\' \'.join(w[\'tags\']),w[\'maker\']]) for w in catalog]\n    vectors=normalize(svd.transform(tf.transform(app_docs)))\n    # Same ID/schema contract as v1. Only app-owned text projected; no automatic source ID merge.\n    artifact=dict(schema=1,model_id=\'kaggle-catalog-tfidf-svd-v2\',feature_schema=\'title-description-tags-maker-v2\',catalog_sha256=hashlib.sha256(json.dumps(catalog,sort_keys=True).encode()).hexdigest(),items=[w[\'id\'] for w in catalog],media=[w[\'medium\'] for w in catalog],vectors=vectors.tolist(),fallback=[w[\'id\'] for w in catalog],training=dict(seed=42,documents=len(corpus),vocabulary=len(tf.vocabulary_),dimensions=64,explained_variance=float(svd.explained_variance_ratio_.sum()),sources={k:v[\'sha256\'] for k,v in sources.items()}),status=\'preview-content-only; aggregate regressions are not personalization quality\')\n    (output/\'catalog-model.json\').write_text(json.dumps(artifact,indent=2))\n    np.savez_compressed(output/\'content-transform.npz\',idf=tf.idf_,components=svd.components_)\n    (output/\'content-vocabulary.json\').write_text(json.dumps(tf.vocabulary_,default=int))\n    # Export aggregate ranking tables for research, not to override personal taste in the app.\n    movie_quality,mp=weighted_scores(movies,\'vote_average\',\'vote_count\')\n    hltb_quality,hp=weighted_scores(hltb,\'game_review_score\',\'game_count_review\')\n    movies.assign(weighted_score=movie_quality)[[\'title\',\'year\',\'vote_average\',\'vote_count\',\'weighted_score\']].sort_values(\'weighted_score\',ascending=False).to_csv(output/\'movie-weighted-ranking.csv\',index=False)\n    hltb.assign(weighted_score=hltb_quality)[[\'game_game_id\',\'game_game_name\',\'game_review_score\',\'game_count_review\',\'game_comp_main\',\'game_comp_main_count\',\'weighted_score\']].sort_values(\'weighted_score\',ascending=False).to_csv(output/\'game-weighted-ranking.csv\',index=False)\n    proposals=[]\n    for w in catalog:\n        frame=hltb if w[\'medium\']==\'game\' else movies if w[\'medium\']==\'movie\' else None\n        matches=frame[frame[\'key\'].eq(title_key(w[\'title\'])) & frame[\'year\'].eq(w[\'year\'])] if frame is not None else pd.DataFrame()\n        proposals.append(dict(app_id=w[\'id\'],title=w[\'title\'],year=w[\'year\'],candidate_count=len(matches),policy=\'title/year candidate only; not canonical approval\'))\n    semantic=dict(**artifact[\'training\'],matrix_shape=list(X.shape),source_documents=dict(movies=len(movies),sales=len(sales_docs),hltb=len(hltb_selected)),app_projection_nonzero=int(np.count_nonzero(np.linalg.norm(vectors,axis=1))),examples=[])\n    for i,w in enumerate(catalog):\n        order=np.argsort(-(vectors@vectors[i]),kind=\'stable\')\n        semantic[\'examples\'].append(dict(seed=w[\'id\'],recommendations=[dict(id=catalog[j][\'id\'],cosine=float(vectors[i]@vectors[j])) for j in order if j!=i][:4]))\n    report=dict(run_id=\'kaggle-metadata-20261116\',runtime=\'Google Colab CPU via colab CLI\',sources=sources,audit=audit,regressions=regressions,semantic=semantic,weighted_score_parameters=dict(movies=mp,hltb=hp),app_identity_proposals=proposals,environment=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,kagglehub=kagglehub.__version__),seconds=time.time()-start,limitations=[\'No per-user events: collaborative training or Recall/NDCG taste claims unsupported.\',\'Aggregate-regression targets are contemporaneous snapshots; hash-group splits are not future forecasting.\',\'Content corpus fitting is unsupervised/transductive; probe cosine is not accuracy.\',\'Year/title proposals can still conflate editions; no source IDs promoted to canonical app works.\',\'HLTB zero values and repeated platform profiles need filtering; durations exported as source raw values, not silently assumed hours.\',\'Movie genre dictionary has no per-film links; credits and genres from reference notebook are unavailable.\'])\n    (output/\'report.json\').write_text(json.dumps(report,indent=2))\n    (output/\'environment.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    import zipfile\n    with zipfile.ZipFile(root/\'results.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for f in output.iterdir():z.write(f,f.name)\n    print(json.dumps(report,indent=2),flush=True)\n    return report\n\nif __name__==\'__main__\':run()\n', 'catalog-metadata.json': '[\n  {\n    "id": "it-takes-two",\n    "title": "It Takes Two",\n    "year": 2021,\n    "medium": "game",\n    "description": "Two people, one extraordinary world. Play as Cody and May, a couple transformed into dolls, on a wonderfully unpredictable journey built entirely for two.",\n    "tags": [\n      "Co-op",\n      "Adventure",\n      "A little wonder"\n    ],\n    "maker": "Hazelight Studios"\n  },\n  {\n    "id": "cyberpunk",\n    "title": "Cyberpunk 2077",\n    "year": 2020,\n    "medium": "game",\n    "description": "Become V, a mercenary searching for a one-of-a-kind implant in Night City. A story about identity, impossible choices, and the people you find at the edge of the future.",\n    "tags": [\n      "Cyberpunk",\n      "Open world",\n      "Night city"\n    ],\n    "maker": "CD PROJEKT RED"\n  },\n  {\n    "id": "elden-ring",\n    "title": "Elden Ring",\n    "year": 2022,\n    "medium": "game",\n    "description": "Rise, Tarnished, and follow the grace of gold into the Lands Between. A vast, mysterious world where every ruin has a story and every distant light is an invitation.",\n    "tags": [\n      "Dark fantasy",\n      "Exploration",\n      "RPG"\n    ],\n    "maker": "FromSoftware"\n  },\n  {\n    "id": "disco-elysium",\n    "title": "Disco Elysium",\n    "year": 2019,\n    "medium": "game",\n    "description": "You are a detective with a city to unravel and a mind that will not stay quiet. Question the world, question yourself, and find something worth holding on to in Revachol.",\n    "tags": [\n      "Detective",\n      "Narrative",\n      "Weird cities"\n    ],\n    "maker": "ZA/UM"\n  },\n  {\n    "id": "baldurs-gate",\n    "title": "Baldur’s Gate 3",\n    "year": 2023,\n    "medium": "game",\n    "description": "Gather your party and return to the Forgotten Realms. An adventure shaped by friendship, freedom, and the choices that make a story yours.",\n    "tags": [\n      "RPG",\n      "Co-op",\n      "Fantasy"\n    ],\n    "maker": "Larian Studios"\n  },\n  {\n    "id": "outer-wilds",\n    "title": "Outer Wilds",\n    "year": 2019,\n    "medium": "game",\n    "description": "An open-world mystery about a solar system trapped in an endless time loop. Pack your curiosity, follow an unfamiliar signal, and discover what waits beyond the next horizon.",\n    "tags": [\n      "Exploration",\n      "Space",\n      "Mystery"\n    ],\n    "maker": "Mobius Digital"\n  },\n  {\n    "id": "exit-8",\n    "title": "The Exit 8",\n    "year": 2023,\n    "medium": "game",\n    "description": "You are trapped in an endless underground passageway. Observe your surroundings carefully. If you find an anomaly, turn back. Find the way to Exit 8.",\n    "tags": [\n      "Liminal",\n      "Horror",\n      "Short journey"\n    ],\n    "maker": "KOTAKE CREATE"\n  },\n  {\n    "id": "dune",\n    "title": "Dune: Part Two",\n    "year": 2024,\n    "medium": "movie",\n    "description": "Paul Atreides joins Chani and the Fremen while seeking revenge against the conspirators who destroyed his family. In the desert, a personal journey becomes something much larger.",\n    "tags": [\n      "Science fiction",\n      "Epic",\n      "World-building"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "blade-runner",\n    "title": "Blade Runner 2049",\n    "year": 2017,\n    "medium": "movie",\n    "description": "A new blade runner uncovers a secret that could change what remains of society. His search leads through a rain-soaked future, toward questions about memory and being human.",\n    "tags": [\n      "Cyberpunk",\n      "Atmospheric",\n      "Identity"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "arrival",\n    "title": "Arrival",\n    "year": 2016,\n    "medium": "movie",\n    "description": "When mysterious spacecraft appear across the world, a linguist searches for a way to communicate. A quiet, moving story about language, time, and the connections that shape a life.",\n    "tags": [\n      "Science fiction",\n      "Language",\n      "Contemplative"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "severance",\n    "title": "Severance",\n    "year": 2022,\n    "medium": "tv",\n    "description": "A team of office workers have surgically divided their memories between work and personal life. When the boundary begins to unravel, they discover that nothing about their work is ordinary.",\n    "tags": [\n      "Mystery",\n      "Identity",\n      "Uncanny"\n    ],\n    "maker": "Dan Erickson"\n  },\n  {\n    "id": "edgerunners",\n    "title": "Cyberpunk: Edgerunners",\n    "year": 2022,\n    "medium": "tv",\n    "description": "A street kid tries to survive in a city obsessed with technology and body modification. A standalone story in the world of Cyberpunk 2077, about found family and the price of a dream.",\n    "tags": [\n      "Anime",\n      "Cyberpunk",\n      "Found family"\n    ],\n    "maker": "Studio Trigger"\n  }\n]'}
for name,content in files.items():(root/name).write_text(content)
sys.path.insert(0,str(root))

In [ ]:
# @title Preprocess, train, select on validation, and evaluate
subprocess.check_call([sys.executable,str(root/"metadata_experiment.py")])
report=json.loads((root/"artifacts/report.json").read_text())

In [ ]:
# @title Inspect scores and app content recommendations
for name,result in report["regressions"].items():
    print(name,"parameters",result["alpha"],"test",result["test"],"baseline",result["train_mean_baseline"])
print(json.dumps(report["semantic"]["examples"],indent=2))
print("Download",root/"results.zip","before stopping the runtime.")

movies parameters 1.0 test {'mae': 0.675149811640086, 'rmse': 0.8559876775856086} baseline {'mae': 0.7305988627736025, 'rmse': 0.9085418996364045}
sales parameters 1.0 test {'mae': 9.309318147553304, 'rmse': 11.869799738949224} baseline {'mae': 11.695556488053322, 'rmse': 14.463009597808142}
hltb parameters 1.0 test {'mae': 7.908027001919783, 'rmse': 10.412276967777231} baseline {'mae': 8.73511060431801, 'rmse': 11.469550287345225}
[
  {
    "seed": "it-takes-two",
    "recommendations": [
      {
        "id": "baldurs-gate",
        "cosine": 0.7333635086231407
      },
      {
        "id": "dune",
        "cosine": 0.6595192026301282
      },
      {
        "id": "elden-ring",
        "cosine": 0.6108480189164011
      },
      {
        "id": "outer-wilds",
        "cosine": 0.5231015047573405
      }
    ]
  },
  {
    "seed": "cyberpunk",
    "recommendations": [
      {
        "id": "edgerunners",
        "cosine": 0.8662009204114998
      },
      {
        "id": "disco-elys